In [0]:
%run ../07_Common/00_setup

In [0]:
df_silver_carts = spark.table("workspace.silver.carts")
print(f" Carritos en Silver: {df_silver_carts.count()}")

In [0]:
df_exploded = (
    df_silver_carts
    .select(
        F.col("id").alias("cart_id"),
        F.col("user_id"),       
        F.explode("products").alias("product_line")
    )
)

print(f"📊 Líneas de producto tras explode: {df_exploded.count()}")

In [0]:

df_fact_cart_items = (
    df_exploded
    .select(
        F.col("cart_id"),
        F.col("user_id"),
        F.col("product_line.id").alias("product_id"),
        F.col("product_line.title").alias("product_title"),
        F.col("product_line.quantity").cast("int").alias("quantity"),
        F.col("product_line.price").cast("decimal(10,2)").alias("price"),
        F.col("product_line.discountPercentage").cast("decimal(5,2)").alias("discount_percentage"),
        F.col("product_line.total").cast("decimal(10,2)").alias("total"),
        F.col("product_line.discountedTotal").cast("decimal(10,2)").alias("discounted_total"),
    )
)

print(f" Registros finales en fact_cart_items: {df_fact_cart_items.count()}")

In [0]:
try:
    (
        df_fact_cart_items.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable("workspace.gold.fact_cart_items")
    )
    estado_final = "success"
    print("Escritura exitosa en gold.fact_cart_items")
except Exception as e:
    estado_final = "failed"
    print(f"ERROR: {e}")

spark.sql(f"""
    UPDATE workspace.control.gold_aggregation_config
    SET last_run_status = '{estado_final}'
    WHERE entity_name = 'fact_cart_items'
""")
dbutils.notebook.exit(f"{estado_final.upper()} | fact_cart_items")
